In [6]:
import pandas as pd
from collections import Counter
import re
from fractions import Fraction
import random
import matplotlib.pyplot as plt
import numpy as np
import math
import time
from feval_ttc import load, DatasetType, LLMType
from typing import List, Dict, Any, Optional, Callable, Tuple
from scipy.special import betainc

# Normalizing the answers
def normalize_answer(ans):
    """Normalize an answer string for equality comparison.
    Handles LaTeX \\frac{...}{...}, plain fractions a/b, and numeric strings
    (collapsing 18.00 -> '18', 36/2 -> '18', '8:00' -> '8'). Returns None for
    empty/None input; otherwise returns a canonical string.
    """
    if ans is None:
        return None

    s = str(ans).strip()
    if s == "":
        return None

    # For: "\frac{36}{2}", "\\frac{ 36 }{ 2 }"
    latex_pattern = r'\\+frac\s*\{\s*([+-]?\d+(?:\.\d+)?)\s*\}\s*\{\s*([+-]?\d+(?:\.\d+)?)\s*\}'
    latex_matches = list(re.finditer(latex_pattern, s))
    if latex_matches:
        # Match the last ones
        m = latex_matches[-1]
        num_str = m.group(1)
        den_str = m.group(2)
        try:
            frac = Fraction(num_str) / Fraction(den_str)
        except ZeroDivisionError:
            return None

        if frac.denominator == 1:
            return str(frac.numerator)
        else:
            return f"{frac.numerator}/{frac.denominator}"

    # For: "36/2"、"-3/5"
    m_frac = re.fullmatch(r'([+-]?\d+(?:\.\d+)?)\s*/\s*([+-]?\d+(?:\.\d+)?)', s)
    if m_frac:
        num_str = m_frac.group(1)
        den_str = m_frac.group(2)
        try:
            frac = Fraction(num_str) / Fraction(den_str)
        except ZeroDivisionError:
            return None

        if frac.denominator == 1:
            return str(frac.numerator)
        else:
            return f"{frac.numerator}/{frac.denominator}"

    # For: string + number
    m_num = re.search(r'[-+]?\d+(\.\d+)?', s)
    if not m_num:
        # If no numbers
        return s.lower()

    num_str = m_num.group(0)
    try:
        val = float(num_str)
    except ValueError:
        return s.lower()

    if val.is_integer():
        return str(int(val))      # 18.0 -> '18'
    else:
        # remove additional 0
        return f"{val:.10f}".rstrip('0').rstrip('.')
    
# Load and preprocess the dataset
# Build per-question "answer probability matrix"
def compute_answer_probability_tables_with_meta(
    dataset_name: str,
    model_classes: dict,
    api_path: str = "api_responses.zip",
    N_samples: int = 40,
):

    all_llm_types = [lt for fam in model_classes.values() for lt in fam]

    try:
        ds_type = DatasetType[dataset_name]
    except KeyError:
        ds_type = next(dt for dt in DatasetType if dt.name.lower() == dataset_name.lower())

    dataset, llm_fns = load(ds_type, all_llm_types, api_path=api_path)
    dataset = list(dataset)

    llm_map = {llm_type: fn for llm_type, fn in zip(all_llm_types, llm_fns)}
    results = {}

    for question_id, dataentry in dataset:
        llm_counters = {}
        all_answers = set()

        answers_seq = {}    
        priors_sorted = {}   

        for family, llm_list in model_classes.items():
            for llm_type in llm_list:
                llm_fn = llm_map[llm_type]
                resp = llm_fn(question_id, N=N_samples)

                #  If not None: do normalize
                seq = []
                for cot in resp.cots[:N_samples]:
                    if cot.answer is None:
                        seq.append(None)
                    else:
                        seq.append(normalize_answer(cot.answer))

                llm_name = llm_type.name
                answers_seq[llm_name] = seq

                # Counter（Include None）
                cnt = Counter(seq)
                llm_counters[llm_name] = cnt
                all_answers.update(cnt.keys())

                probs = [c / N_samples for c in cnt.values()]
                priors_sorted[llm_name] = sorted(probs, reverse=True)

        # All unique answers for this question (across all LLMs), includes None
        all_answers = sorted(all_answers, key=lambda x: str(x))
        all_answer_strs = [str(a) for a in all_answers]  

        # Build probability DataFrame: rows = LLM, cols = answer probabilities
        rows_dict = {}
        for llm_name, cnt in llm_counters.items():
            row = {
                ans_str: cnt.get(ans, 0) / N_samples
                for ans, ans_str in zip(all_answers, all_answer_strs)
            }
            rows_dict[llm_name] = row

        df_q = pd.DataFrame.from_dict(rows_dict, orient="index")
        df_q.index.name = "LLM"

        true_answer_norm = normalize_answer(dataentry.answer)
        
        true_mode = {}
        for llm_name, seq in answers_seq.items():
            true_mode[llm_name] = Counter(seq).most_common(1)[0][0]
        
        results[question_id] = {
            "question": dataentry.question,
            "true_answer": dataentry.answer,
            "true_answer_norm": true_answer_norm,
            "prob_table": df_q,
            "answers_seq": answers_seq,
            "priors_sorted": priors_sorted,
            "true_mode": true_mode,
        }

    return results

def bootstrap_answer_probability_tables_with_meta(
    old_results: dict,
    N_samples_new: int,
    seed: int = 0,
):
    """
    Bootstrap (with replacement) from the empirical distribution induced by the
    original 40 samples per (question, LLM). Returns a new results dict with the
    same structure as compute_answer_probability_tables_with_meta.
    """
    rng = np.random.default_rng(seed)

    new_results = {}

    # Keep question iteration order as in old_results 
    for question_id, q in old_results.items():
        answers_seq_old = q["answers_seq"]   # dict: llm_name -> list (length old N, incl None)
        llm_names = list(answers_seq_old.keys())  # preserve order exactly

        answers_seq_new = {}     # llm_name -> new bootstrapped list length N_samples_new
        llm_counters_new = {}    # llm_name -> Counter(new seq)
        all_answers = set()      # union of answers across llms for this question

        # For each LLM: build empirical distribution from old 40 seq, then bootstrap sample
        for llm_name in llm_names:
            seq_old = answers_seq_old[llm_name]
            cnt_old = Counter(seq_old)

            # Empirical support and probs (order doesn't matter for sampling)
            support = list(cnt_old.keys())
            probs = np.array([cnt_old[a] for a in support], dtype=float)
            probs /= probs.sum()  # sum to 1

            # Bootstrap draw from empirical distribution
            sampled = rng.choice(support, size=N_samples_new, replace=True, p=probs).tolist()

            answers_seq_new[llm_name] = sampled

            cnt_new = Counter(sampled)
            llm_counters_new[llm_name] = cnt_new
            all_answers.update(cnt_new.keys())

        # Build prob_table with consistent columns across LLMs
        all_answers = sorted(all_answers, key=lambda x: str(x))
        all_answer_strs = [str(a) for a in all_answers]

        rows_dict = {}
        for llm_name in llm_names:
            cnt_new = llm_counters_new[llm_name]
            row = {
                ans_str: cnt_new.get(ans, 0) / N_samples_new
                for ans, ans_str in zip(all_answers, all_answer_strs)
            }
            rows_dict[llm_name] = row

        df_q = pd.DataFrame.from_dict(rows_dict, orient="index")
        df_q.index.name = "LLM"

        # priors_sorted for the new bootstrapped distribution (descending)
        priors_sorted_new = {}
        for llm_name in llm_names:
            cnt_new = llm_counters_new[llm_name]
            probs_new = [c / N_samples_new for c in cnt_new.values()]
            priors_sorted_new[llm_name] = sorted(probs_new, reverse=True)

        # Pack the same fields as old_results (keep question + true answer metadata unchanged)
        new_results[question_id] = {
            "question": q["question"],
            "true_answer": q["true_answer"],
            "true_answer_norm": q["true_answer_norm"],
            "prob_table": df_q,
            "answers_seq": answers_seq_new,
            "priors_sorted": priors_sorted_new,
        }

    return new_results


# First split the questions in the dataset: use the training dataset to learn the prior,
def split_results_train_test(results, train_ratio=0.7, random_state=42):
    """
    Split results (dict[qid]->obj) into train/test by question_id.

    Returns:
        train_results, test_results
    """
    qids = list(results.keys())
    rng = random.Random(random_state)
    rng.shuffle(qids)

    split_idx = int(len(qids) * train_ratio)
    train_qids = qids[:split_idx]
    test_qids  = qids[split_idx:]

    train_results = {qid: results[qid] for qid in train_qids}
    test_results  = {qid: results[qid] for qid in test_qids}

    return train_results, test_results


def _vec_from(x):
    return list(x.values()) if isinstance(x, dict) else list(x)

def _prior_to_prob_list(prior_obj):
    """
    Convert one question's prior into a list of probabilities in descending order.
    """
    if prior_obj is None:
        return []
    if isinstance(prior_obj, dict):
        probs = list(prior_obj.values())
        probs.sort(reverse=True)
        return probs
    if isinstance(prior_obj, (list, tuple)):
        if len(prior_obj) == 0:
            return []
        # list of floats
        if isinstance(prior_obj[0], (int, float)):
            probs = list(prior_obj)
            probs.sort(reverse=True)
            return probs
        # list of (answer, prob)
        if isinstance(prior_obj[0], (list, tuple)) and len(prior_obj[0]) >= 2:
            probs = [x[1] for x in prior_obj]
            probs.sort(reverse=True)
            return probs
    raise TypeError(f"Unsupported prior format: {type(prior_obj)}")


def build_prior_cache_from_train_results(train_results, llm_name, priors_key="priors_sorted"):
    """
    Build prior_cache from train_results.

    Returns:
        prior_cache = {"K": K, "P_list": P_list}
    """
    raw_vecs = []
    K = 0

    for qid, item in train_results.items():
        prior_obj = item[priors_key][llm_name]
        v = _prior_to_prob_list(prior_obj)  # sorted desc
        raw_vecs.append(v)
        K = max(K, len(v))

    P_list = []
    for v in raw_vecs:
        if len(v) < K:
            v = v + [0.0] * (K - len(v))
        s = sum(v)
        if s > 0:
            v = [x / s for x in v]
        P_list.append(v)

    return {"K": K, "P_list": P_list}

def ac_beta_confidence(v1: int, v2: int) -> float:
    return float(betainc(v2 + 1, v1 + 1, 0.5))

def stop_AC_beta(answers_all, conf_thre: float = 0.95):
    """
    ASC (Beta Stopping)
    """
    start_time = time.time()
    cnt = Counter()
    
    confidence = 0.0
    final_answer = None
    n_stop = 0
    
    for t, ans in enumerate(answers_all, start=1):
        cnt[ans] += 1
        
        top2 = cnt.most_common(2)
        v1 = top2[0][1]
        v2 = top2[1][1] if len(top2) > 1 else 0
        
        confidence = ac_beta_confidence(v1, v2)
        final_answer = top2[0][0]
        n_stop = t
        
        if confidence >= conf_thre:
            break
            
    return {
        "n_stop": n_stop,
        "final_answer": final_answer,
        "confidence": confidence,
        "time": time.time() - start_time
    }

In [7]:
def log_add(log_x: float, log_y: float) -> float:
    """
    Computes log(exp(log_x) + exp(log_y)) safely.
    """
    if log_x == float('-inf'): return log_y
    if log_y == float('-inf'): return log_x
    
    if log_x > log_y:
        return log_x + math.log1p(math.exp(log_y - log_x))
    else:
        return log_y + math.log1p(math.exp(log_x - log_y))


# ============================================================

def _compute_log_S_psi_algo2_strict(prior_tail: List[float], 
                                    n_L_bar: int, 
                                    v_d: int, 
                                    c_prime_d: int) -> float:
    """
    Implementation of Algorithm 2 in log-space
    """
    N = len(prior_tail)
    
    # Determine m_min
    if v_d > 0:
        m_min = min(N, n_L_bar // v_d)
    else:
        # If cutoff is 0, elements can only be 0.
        # This contributes to 'Tie' (m) if they are 0.
        # But if residual > 0, it's impossible to sum to n_L_bar with 0s.
        if n_L_bar > 0: return float('-inf')
        m_min = N 

    # Initialize DP Table A[m][t] (Log Probabilities)
    # A[m][t] = -inf (0.0 prob) everywhere
    A = [[float('-inf')] * (n_L_bar + 1) for _ in range(m_min + 1)]
    # Base case: A[0][0] = 0.0 (log 1.0)
    A[0][0] = 0.0

    # Pre-compute log factorials for g_j calculation
    log_fact = [0.0] * (v_d + 1)
    for i in range(2, v_d + 1):
        log_fact[i] = math.lgamma(i + 1)

    # DP Loop
    for p_j in prior_tail:
        if p_j <= 0:
            log_p_j = float('-inf')
        else:
            log_p_j = math.log(p_j)

        # Initialize B to -inf
        B = [[float('-inf')] * (n_L_bar + 1) for _ in range(m_min + 1)]
        
        # Precompute log_g_j[r]
        log_g_j = [float('-inf')] * (v_d + 1)
        if log_p_j == float('-inf'):
            log_g_j[0] = 0.0 
        else:
            for r in range(v_d + 1):
                log_g_j[r] = (r * log_p_j) - log_fact[r]

        for m in range(m_min + 1):
            for t in range(n_L_bar + 1):
                log_A_curr = A[m][t]
                if log_A_curr == float('-inf'):
                    continue
                
                # Case 1: Non-tie transition (r < v_d)
                r_limit = min(v_d - 1, n_L_bar - t)
                for r in range(r_limit + 1):
                    # B += A * g  => logB = log_add(logB, logA + logg)
                    term = log_A_curr + log_g_j[r]
                    B[m][t + r] = log_add(B[m][t + r], term)
                
                # Case 2: Tie transition (r = v_d)
                if (t + v_d <= n_L_bar) and (m + 1 <= m_min):
                    term = log_A_curr + log_g_j[v_d]
                    B[m + 1][t + v_d] = log_add(B[m + 1][t + v_d], term)
        
        A = B

    # Apply Weights and Sum
    log_S_psi = float('-inf')
    
    for m in range(m_min + 1):
        log_val = A[m][n_L_bar]
        if log_val == float('-inf'):
            continue
            
        # Weight = 1 / binom(c'_d + m, c'_d)
        binom_val = math.comb(c_prime_d + m, c_prime_d)
        log_weight = -math.log(binom_val)
        #log_binom = math.lgamma(c_prime_d + m + 1) - math.lgamma(c_prime_d + 1) - math.lgamma(m + 1)
        #log_weight = -log_binom
        
        term = log_val + log_weight
        log_S_psi = log_add(log_S_psi, term)
        
    # Final Scaling: Multiply by n_L_bar!
    if log_S_psi != float('-inf'):
        log_fact_n = math.lgamma(n_L_bar + 1)
        log_S_psi += log_fact_n
    
    return log_S_psi

###############################################################
# Next: For L=2
###############################################################
def compute_all_S_tilde_L2(K: int, p_values: List[float], n: int, n_1: int) -> List[float]:
    psum = sum(p_values)
    if psum <= 0: return [float('-inf')] * K
    p = [x / psum for x in p_values]
    
    n_L_bar = n - n_1
    v_d = n_1
    c_prime_d = 1 
    
    log_S_results = []
    for i in range(K):
        prior_tail = p[:i] + p[i+1:]
        val = _compute_log_S_psi_algo2_strict(prior_tail, n_L_bar, v_d, c_prime_d)
        log_S_results.append(val)
    return log_S_results

def L2_posterior_tilde(answers: List[str], prior: List[float]) -> float:
    n = len(answers)
    if n == 0: return 0.0
    cnt = Counter(answers)
    n_1 = cnt.most_common(1)[0][1] if cnt else 0
        
    K = len(prior)
    log_S_tilde = compute_all_S_tilde_L2(K, prior, n, n_1)
    
    psum = sum(prior)
    p = [x / psum for x in prior]
    
    log_terms = []
    for i in range(K):
        if p[i] > 0 and log_S_tilde[i] != float('-inf'):
            val = n_1 * math.log(p[i]) + log_S_tilde[i]
            log_terms.append(val)
        else:
            log_terms.append(float("-inf"))
            
            
    max_log = max(log_terms)
    if max_log == float("-inf"): return 0.0
    
    denom = sum(math.exp(lt - max_log) for lt in log_terms)
    prob = math.exp(max_log - (max_log + math.log(denom))) 
    return max(0.0, min(1.0, prob))


###############################################################


def stop_L2_tilde(answers_all: List[str], prior: List[float], prob_thre: float) -> Dict[str, Any]:
    start_time = time.time()
    stopped = False
    posterior = 0.0
    n_idx = 0
    prior_sorted = sorted(prior, reverse=True)
    
    for n_idx in range(1, len(answers_all) + 1):
        answers_n = answers_all[:n_idx]
        posterior = L2_posterior_tilde(answers_n, prior_sorted)
        if posterior >= prob_thre:
            stopped = True
            break
            
    time_spend = time.time() - start_time
    final_answers = answers_all[:n_idx] if len(answers_all) > 0 else []
    mode = Counter(final_answers).most_common(1)[0][0] if final_answers else None
    
    return {"n_stop": n_idx, "final_answer": mode, "posterior": posterior, "stopped": stopped, "time": time_spend}



###############################################################
# Next: For L=3
###############################################################

def compute_all_S_tilde_ij_L3(K: int, prior: List[float], n: int, n_1: int, n_2: int) -> Dict[Tuple[int, int], float]:
    psum = sum(prior)
    if psum <= 0: return {}
    p = [x / psum for x in prior]
    
    n_L_bar = n - n_1 - n_2
    v_d = n_2 
    c_prime_d = 2 if n_1 == n_2 else 1
        
    log_S_ij_results = {}
    for i in range(K):
        for j in range(K):
            if i == j: continue
            prior_tail = [p[k] for k in range(K) if k != i and k != j]
            val = _compute_log_S_psi_algo2_strict(prior_tail, n_L_bar, v_d, c_prime_d)
            log_S_ij_results[(i, j)] = val
    return log_S_ij_results

def L3_posterior_tilde(answers: List[str], prior: List[float]) -> float:
    n = len(answers)
    if n == 0: return 0.0

    cnt = Counter(answers)
    unique_count = len(cnt)
    
    # IF less than 2 unique answer: back to L=2
    if unique_count < 2:
        return L2_posterior_tilde(answers, prior)
    # ======================

    counts = [c for ans, c in cnt.most_common()]
    # Fallback guarantees we have at least 2 counts
    n_1, n_2 = counts[0], counts[1]
    
    K = len(prior)
    psum = sum(prior)
    p = [x / psum for x in prior]
    
    log_S_ij = compute_all_S_tilde_ij_L3(K, prior, n, n_1, n_2)
    log_likelihoods_i = []
    
    for i in range(K):
        if p[i] <= 0:
            log_likelihoods_i.append(float("-inf"))
            continue
            
        terms_j = [] 
        for j in range(K):
            if i == j or p[j] <= 0: continue
            l_s = log_S_ij.get((i, j), float('-inf'))
            if l_s != float('-inf'):
                term = n_1 * math.log(p[i]) + n_2 * math.log(p[j]) + l_s
                terms_j.append(term)
        
        if not terms_j:
            log_likelihoods_i.append(float("-inf"))
        else:
            mx = max(terms_j)
            sum_exp = sum(math.exp(t - mx) for t in terms_j)
            log_likelihoods_i.append(mx + math.log(sum_exp))

    max_total = max(log_likelihoods_i)
    if max_total == float("-inf"): return 0.0
    
    denom = sum(math.exp(l - max_total) for l in log_likelihoods_i)
    prob = math.exp(max_total - (max_total + math.log(denom)))
    return max(0.0, min(1.0, prob))


def stop_L3_tilde(answers_all: List[str], prior: List[float], prob_thre: float) -> Dict[str, Any]:
    start_time = time.time()
    stopped = False
    posterior = 0.0
    n_idx = 0
    prior_sorted = sorted(prior, reverse=True)
    
    for n_idx in range(1, len(answers_all) + 1):
        answers_n = answers_all[:n_idx]
        posterior = L3_posterior_tilde(answers_n, prior_sorted)
        if posterior >= prob_thre:
            stopped = True
            break
            
    time_spend = time.time() - start_time
    final_answers = answers_all[:n_idx] if len(answers_all) > 0 else []
    mode = Counter(final_answers).most_common(1)[0][0] if final_answers else None
    
    return {"n_stop": n_idx, "final_answer": mode, "posterior": posterior, "stopped": stopped, "time": time_spend}


###############################################################
# Next: For L=4
###############################################################

def compute_all_S_tilde_ijk_L4(K: int, prior: List[float], n: int, n_1: int, n_2: int, n_3: int) -> Dict[Tuple[int, int, int], float]:
    psum = sum(prior)
    if psum <= 0: return {}
    p = [x / psum for x in prior]
    
    n_L_bar = n - n_1 - n_2 - n_3
    v_d = n_3
    head_counts = [n_1, n_2, n_3]
    c_prime_d = head_counts.count(v_d)
    
    log_S_ijk_results = {}
    for i in range(K):
        for j in range(K):
            if i == j: continue
            for k in range(K):
                if k == i or k == j: continue
                prior_tail = [p[idx] for idx in range(K) if idx not in [i, j, k]]
                val = _compute_log_S_psi_algo2_strict(prior_tail, n_L_bar, v_d, c_prime_d)
                log_S_ijk_results[(i, j, k)] = val
    return log_S_ijk_results

def L4_posterior_tilde(answers: List[str], prior: List[float]) -> float:
    n = len(answers)
    if n == 0: return 0.0
    
    # === FALLBACK LOGIC ===
    cnt = Counter(answers)
    unique_count = len(cnt)
    
    if unique_count < 3:
        return L3_posterior_tilde(answers, prior)
    # ======================

    counts = [c for ans, c in cnt.most_common()]
    n_1, n_2, n_3 = counts[0], counts[1], counts[2]
    
    K = len(prior)
    psum = sum(prior)
    p = [x / psum for x in prior]
    
    log_S_ijk = compute_all_S_tilde_ijk_L4(K, prior, n, n_1, n_2, n_3)
    log_likelihoods_i = []
    
    for i in range(K):
        if p[i] <= 0:
            log_likelihoods_i.append(float("-inf"))
            continue
            
        terms_jk = []
        for j in range(K):
            if i == j or p[j] <= 0: continue
            for k in range(K):
                if k == i or k == j or p[k] <= 0: continue
                l_s = log_S_ijk.get((i, j, k), float('-inf'))
                if l_s != float('-inf'):
                    term = n_1 * math.log(p[i]) + n_2 * math.log(p[j]) + n_3 * math.log(p[k]) + l_s
                    terms_jk.append(term)
                    
        if not terms_jk:
            log_likelihoods_i.append(float("-inf"))
        else:
            mx = max(terms_jk)
            sum_exp = sum(math.exp(t - mx) for t in terms_jk)
            log_likelihoods_i.append(mx + math.log(sum_exp))

    max_total = max(log_likelihoods_i)
    if max_total == float("-inf"): return 0.0
    
    denom = sum(math.exp(l - max_total) for l in log_likelihoods_i)
    prob = math.exp(max_total - (max_total + math.log(denom)))
    return max(0.0, min(1.0, prob))

def stop_L4_tilde(answers_all: List[str], prior: List[float], prob_thre: float) -> Dict[str, Any]:
    start_time = time.time()
    stopped = False
    posterior = 0.0
    n_idx = 0
    prior_sorted = sorted(prior, reverse=True)
    
    for n_idx in range(1, len(answers_all) + 1):
        answers_n = answers_all[:n_idx]
        posterior = L4_posterior_tilde(answers_n, prior_sorted)
        if posterior >= prob_thre:
            stopped = True
            break
            
    time_spend = time.time() - start_time
    final_answers = answers_all[:n_idx] if len(answers_all) > 0 else []
    mode = Counter(final_answers).most_common(1)[0][0] if final_answers else None
    
    return {"n_stop": n_idx, "final_answer": mode, "posterior": posterior, "stopped": stopped, "time": time_spend}

In [8]:
def split_results_train_test_limited(results, train_ratio=0.7, random_state=42):
    """
    Split results into train/test by question_id.
    """
    qids = list(results.keys())
    rng = random.Random(random_state)
    rng.shuffle(qids)

    split_idx = int(len(qids) * train_ratio)
    train_qids = qids[:split_idx]
    test_qids  = qids[split_idx:]


    train_qids = train_qids[:250]
    test_qids  = test_qids[:250]

    train_results = {qid: results[qid] for qid in train_qids}
    test_results  = {qid: results[qid] for qid in test_qids}

    return train_results, test_results


In [9]:
def run_stop_per_llm_known(train_results, test_results, prob_thre, L_approx, llm_names=None):
    test_size = len(test_results)
    
    if llm_names is None:
        first_q = next(iter(test_results.values()))
        llm_names = list(first_q["answers_seq"].keys())

    #prior_cache_by_llm = {}
    #for llm in llm_names:
    #    prior_cache_by_llm[llm] = build_prior_cache_from_train_results(
    #        train_results, llm, priors_key="priors_sorted"
    #    )

    records_by_llm = {llm: [] for llm in llm_names}
    
    start_time = time.time()
    for done, (qid, obj) in enumerate(test_results.items()):
        answers_seq = obj["answers_seq"]
        priors_sorted = obj["priors_sorted"]
        
        for llm in llm_names:
            answers_all = answers_seq[llm]
            prior = priors_sorted[llm]

            if L_approx == 2:
                out = stop_L2_tilde(answers_all, prior, prob_thre)
            elif L_approx == 3:
                out = stop_L3_tilde(answers_all, prior, prob_thre)
            elif L_approx == 4:
                out = stop_L4_tilde(answers_all, prior, prob_thre)    
            else:
                raise ValueError("L_approx must be 2/3/4")
    
            records_by_llm[llm].append({
                "question_id": qid,
                "n_stop": out["n_stop"],
                "final_answer": out["final_answer"],
                "posterior_at_stop": out["posterior"],
                "stopped": out["stopped"],
                "time": out["time"],
            })
            
        if done % 40 == 0 and done > 0:
            print(f"Finished {done}/{test_size} | Time spent: {(time.time()-start_time)/60:.2f} mins")

    tables_by_llm = {}
    for llm, recs in records_by_llm.items():
        tables_by_llm[llm] = pd.DataFrame(recs).sort_values("question_id").set_index("question_id")

    return tables_by_llm




def run_AC_beta_per_llm(results, conf_thre: float = 0.95, llm_names=None):
    first_q = next(iter(results.values()))
    if llm_names is None:
        llm_names = list(first_q["answers_seq"].keys())

    records_by_llm = {llm: [] for llm in llm_names}

    for qid, obj in results.items():
        answers_seq = obj["answers_seq"]
        for llm in llm_names:
            answers_all = answers_seq[llm]
            out = stop_AC_beta(answers_all, conf_thre=conf_thre)

            records_by_llm[llm].append({
                "question_id": qid,
                "n_stop": out["n_stop"],
                "final_answer": out["final_answer"],
                "confidence": out["confidence"],
                "time": out["time"],
            })

    tables_by_llm = {}
    for llm, recs in records_by_llm.items():
        df = pd.DataFrame(recs).sort_values("question_id").set_index("question_id")
        tables_by_llm[llm] = df

    return tables_by_llm


def print_avg_metrics_rp(
    tables_by_llm_or_all,
    results,
    L_approx=None,
    method=None,
    rp=None,        
):
    # normalize input to a list
    if isinstance(tables_by_llm_or_all, list):
        tables_list = tables_by_llm_or_all
        if rp is None:
            rp = len(tables_list)
    else:
        tables_list = [tables_by_llm_or_all]
        if rp is None:
            rp = 1

    if method is None:
        method = f"L={L_approx}" if L_approx is not None else "Unknown"

    # preserve llm order from the first replicate
    llm_order = list(tables_list[0].keys())

    rows = []
    for rep_idx, tables_by_llm in enumerate(tables_list):
        for llm in llm_order:
            df = tables_by_llm[llm]
            for qid, row in df.iterrows():
            
                if qid not in results:
                    continue
                # -------------------

                final_ans = row.get("final_answer", None)
                t = row.get("time", None)
                n_stop = row.get("n_stop", None)

                true_mode = results[qid]["true_mode"][llm]
                true_ans = results[qid]["true_answer_norm"]

                rows.append({
                    "rep": rep_idx,
                    "llm": llm,
                    "qid": qid,
                    "n_stop": n_stop,
                    "time": t,
                    "match_true_mode": 1.0 if final_ans == true_mode else 0.0,
                    "match_true_answer": 1.0 if final_ans == true_ans else 0.0,
                })

    if not rows:
        print(f"Warning: No matching questions found in results for {method}. Check your inputs.")
        return

    df_all = pd.DataFrame(rows)

    # average over all samples
    agg = (df_all.groupby("llm", as_index=False)
                  .agg(
                      mean_n_stop=("n_stop", "mean"),
                      std_n_stop=("n_stop", "std"),
                      count_n_stop=("n_stop", "count"),
                      
                      mean_time=("time", "mean"),
                      
                      mode_accuracy=("match_true_mode", "mean"),
                      std_mode=("match_true_mode", "std"),
                      count_mode=("match_true_mode", "count"),
                      
                      answer_accuracy=("match_true_answer", "mean"),
                      std_answer=("match_true_answer", "std"),
                      count_answer=("match_true_answer", "count"),
                  ))

    agg_map = {r["llm"]: r for _, r in agg.iterrows()}

    print(f"Average metrics per LLM ({method}; rp={rp}):")
    for llm in llm_order:
        r = agg_map.get(llm, None)
        if r is None:
            continue
            
        # 95% CI: 1.96 * std
        def get_ci_95(std, count):
            if pd.isna(std) or count <= 1:
                return 0.0
            return 1.96 * (std / np.sqrt(count))
            
        ci_n_stop = get_ci_95(r['std_n_stop'], r['count_n_stop'])
        ci_mode = get_ci_95(r['std_mode'], r['count_mode'])
        ci_answer = get_ci_95(r['std_answer'], r['count_answer'])

        print(
            f"{llm:20s}  "
            f"mean n_stop = {r['mean_n_stop']:.3f} ± {ci_n_stop:.3f}    "
            f"mode accuracy = {r['mode_accuracy']:.3f} ± {ci_mode:.3f}    "
            f"answer accuracy = {r['answer_accuracy']:.3f} ± {ci_answer:.3f}"
        )
    print()    

In [17]:
# The path for the dataset
api_path_global = "/Users/api_responses_fixed.zip"

rp = 5
tables_by_llm_L2_all = []
tables_by_llm_L3_all = []
tables_by_llm_L4_all = []
tables_by_llm_ACbeta_all = []
model_classes = {
    #"LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        #LLMType.LLaMA405B31,
    #],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ],
}

results = compute_answer_probability_tables_with_meta(
    dataset_name="CommonsenseQA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)


train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )
    
    prob_thre = 0.99
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_all.append(tables_by_llm_L2)
    end_L2 = time.time()
    print('L=2 finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_all.append(tables_by_llm_L3)
    end_L3 = time.time()
    print('L=3 finished! Time=',(end_L3-end_L2)/60,'mins')
    tables_by_llm_L4 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 4)
    tables_by_llm_L4_all.append(tables_by_llm_L4)
    end_L4 = time.time()
    print('L=4 finished! Time=',(end_L4-end_L3)/60,'mins')
    tables_by_llm_ACbeta = run_AC_beta_per_llm(test_results, conf_thre=prob_thre)
    tables_by_llm_ACbeta_all.append(tables_by_llm_ACbeta)
    




Num. of repeat: 0
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=2 finished! Time= 0.0038359840710957844 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=3 finished! Time= 0.000770413875579834 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=4 finished! Time= 0.00048621892929077146 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 12

In [18]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_all, test_results, L_approx=3)
print_avg_metrics_rp(tables_by_llm_L4_all, test_results, L_approx=4)

Prob. Threshold =  0.99
Average metrics per LLM (AC-Beta; rp=5):
Qwen72B25             mean n_stop = 8.044    mode accuracy = 1.000    answer accuracy = 0.876
GPT4oMINI             mean n_stop = 7.556    mode accuracy = 1.000    answer accuracy = 0.860

Average metrics per LLM (L=2; rp=5):
Qwen72B25             mean n_stop = 4.272    mode accuracy = 0.994    answer accuracy = 0.880
GPT4oMINI             mean n_stop = 3.038    mode accuracy = 0.996    answer accuracy = 0.858

Average metrics per LLM (L=3; rp=5):
Qwen72B25             mean n_stop = 4.242    mode accuracy = 0.994    answer accuracy = 0.880
GPT4oMINI             mean n_stop = 3.003    mode accuracy = 0.996    answer accuracy = 0.858

Average metrics per LLM (L=4; rp=5):
Qwen72B25             mean n_stop = 4.242    mode accuracy = 0.994    answer accuracy = 0.880
GPT4oMINI             mean n_stop = 3.003    mode accuracy = 0.996    answer accuracy = 0.858



In [19]:
rp = 5
tables_by_llm_L2_all = []
tables_by_llm_L3_all = []
tables_by_llm_L4_all = []
tables_by_llm_ACbeta_all = []
model_classes = {
    #"LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        #LLMType.LLaMA405B31,
    #],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ],
}

results = compute_answer_probability_tables_with_meta(
    dataset_name="CommonsenseQA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)


train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )
    
    prob_thre = 0.975
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_all.append(tables_by_llm_L2)
    end_L2 = time.time()
    print('L=2 finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_all.append(tables_by_llm_L3)
    end_L3 = time.time()
    print('L=3 finished! Time=',(end_L3-end_L2)/60,'mins')
    tables_by_llm_L4 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 4)
    tables_by_llm_L4_all.append(tables_by_llm_L4)
    end_L4 = time.time()
    print('L=4 finished! Time=',(end_L4-end_L3)/60,'mins')
    tables_by_llm_ACbeta = run_AC_beta_per_llm(test_results, conf_thre=prob_thre)
    tables_by_llm_ACbeta_all.append(tables_by_llm_ACbeta)

Num. of repeat: 0
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=2 finished! Time= 0.0037369489669799804 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=3 finished! Time= 0.000728599230448405 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=4 finished! Time= 0.0004414836565653483 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120

In [20]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_all, test_results, L_approx=3)
print_avg_metrics_rp(tables_by_llm_L4_all, test_results, L_approx=4)

Prob. Threshold =  0.975
Average metrics per LLM (AC-Beta; rp=5):
Qwen72B25             mean n_stop = 6.748    mode accuracy = 1.000    answer accuracy = 0.876
GPT4oMINI             mean n_stop = 6.312    mode accuracy = 1.000    answer accuracy = 0.860

Average metrics per LLM (L=2; rp=5):
Qwen72B25             mean n_stop = 3.742    mode accuracy = 0.994    answer accuracy = 0.880
GPT4oMINI             mean n_stop = 2.838    mode accuracy = 0.996    answer accuracy = 0.858

Average metrics per LLM (L=3; rp=5):
Qwen72B25             mean n_stop = 3.682    mode accuracy = 0.994    answer accuracy = 0.880
GPT4oMINI             mean n_stop = 2.830    mode accuracy = 0.996    answer accuracy = 0.858

Average metrics per LLM (L=4; rp=5):
Qwen72B25             mean n_stop = 3.682    mode accuracy = 0.994    answer accuracy = 0.880
GPT4oMINI             mean n_stop = 2.830    mode accuracy = 0.996    answer accuracy = 0.858



In [21]:
rp = 5
tables_by_llm_L2_all = []
tables_by_llm_L3_all = []
tables_by_llm_L4_all = []
tables_by_llm_ACbeta_all = []
model_classes = {
    #"LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        #LLMType.LLaMA405B31,
    #],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ],
}

results = compute_answer_probability_tables_with_meta(
    dataset_name="CommonsenseQA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)


train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )
    
    prob_thre = 0.95
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_all.append(tables_by_llm_L2)
    end_L2 = time.time()
    print('L=2 finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_all.append(tables_by_llm_L3)
    end_L3 = time.time()
    print('L=3 finished! Time=',(end_L3-end_L2)/60,'mins')
    tables_by_llm_L4 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 4)
    tables_by_llm_L4_all.append(tables_by_llm_L4)
    end_L4 = time.time()
    print('L=4 finished! Time=',(end_L4-end_L3)/60,'mins')
    tables_by_llm_ACbeta = run_AC_beta_per_llm(test_results, conf_thre=prob_thre)
    tables_by_llm_ACbeta_all.append(tables_by_llm_ACbeta)

Num. of repeat: 0
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=2 finished! Time= 0.003639264901479085 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=3 finished! Time= 0.000697469711303711 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=4 finished! Time= 0.00041591326395670574 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120

In [22]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_all, test_results, L_approx=3)
print_avg_metrics_rp(tables_by_llm_L4_all, test_results, L_approx=4)

Prob. Threshold =  0.95
Average metrics per LLM (AC-Beta; rp=5):
Qwen72B25             mean n_stop = 5.284    mode accuracy = 0.996    answer accuracy = 0.872
GPT4oMINI             mean n_stop = 5.096    mode accuracy = 1.000    answer accuracy = 0.860

Average metrics per LLM (L=2; rp=5):
Qwen72B25             mean n_stop = 3.405    mode accuracy = 0.989    answer accuracy = 0.875
GPT4oMINI             mean n_stop = 2.626    mode accuracy = 0.996    answer accuracy = 0.858

Average metrics per LLM (L=3; rp=5):
Qwen72B25             mean n_stop = 3.382    mode accuracy = 0.990    answer accuracy = 0.875
GPT4oMINI             mean n_stop = 2.590    mode accuracy = 0.996    answer accuracy = 0.858

Average metrics per LLM (L=4; rp=5):
Qwen72B25             mean n_stop = 3.382    mode accuracy = 0.990    answer accuracy = 0.875
GPT4oMINI             mean n_stop = 2.590    mode accuracy = 0.996    answer accuracy = 0.858



In [41]:
rp = 5
tables_by_llm_L2_all = []
tables_by_llm_L3_all = []
tables_by_llm_L4_all = []
tables_by_llm_ACbeta_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    #"Qwen": [
        #LLMType.Qwen1B25,
    #    LLMType.Qwen72B25,
    #],
    #"GPT": [
        #LLMType.GPT3,
    #    LLMType.GPT4oMINI,
    #],
}

results = compute_answer_probability_tables_with_meta(
    dataset_name="CommonsenseQA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)


train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )
    
    prob_thre = 0.99
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_all.append(tables_by_llm_L2)
    end_L2 = time.time()
    print('L=2 finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_all.append(tables_by_llm_L3)
    end_L3 = time.time()
    print('L=3 finished! Time=',(end_L3-end_L2)/60,'mins')
    #tables_by_llm_L4 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 4)
    #tables_by_llm_L4_all.append(tables_by_llm_L4)
    #end_L4 = time.time()
    #print('L=4 finished! Time=',(end_L4-end_L3)/60,'mins')
    tables_by_llm_ACbeta = run_AC_beta_per_llm(test_results, conf_thre=prob_thre)
    tables_by_llm_ACbeta_all.append(tables_by_llm_ACbeta)
    

Num. of repeat: 0
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.01 mins
Finished 120/250 | Time spent: 0.01 mins
Finished 160/250 | Time spent: 0.02 mins
Finished 200/250 | Time spent: 0.02 mins
Finished 240/250 | Time spent: 0.02 mins
L=2 finished! Time= 0.022164535522460938 mins
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.01 mins
Finished 120/250 | Time spent: 0.01 mins
Finished 160/250 | Time spent: 0.01 mins
Finished 200/250 | Time spent: 0.02 mins
Finished 240/250 | Time spent: 0.02 mins
L=3 finished! Time= 0.01570348342259725 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.01 mins
Finished 120/250 | Time spent: 0.01 mins
Finished 160/250 | Time spent: 0.01 mins
Finished 200/250 | Time spent: 0.01 mins
Finished 240/250 | Time spent: 0.01 mins
L=2 finished! Time= 0.01203248103459676 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250

In [42]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_all, test_results, L_approx=3)

Prob. Threshold =  0.99
Average metrics per LLM (AC-Beta; rp=5):
LLaMA405B31           mean n_stop = 12.032    mode accuracy = 1.000    answer accuracy = 0.900

Average metrics per LLM (L=2; rp=5):
LLaMA405B31           mean n_stop = 8.705    mode accuracy = 0.990    answer accuracy = 0.894

Average metrics per LLM (L=3; rp=5):
LLaMA405B31           mean n_stop = 8.186    mode accuracy = 0.990    answer accuracy = 0.894



In [43]:
rp = 5
tables_by_llm_L2_all = []
tables_by_llm_L3_all = []
tables_by_llm_L4_all = []
tables_by_llm_ACbeta_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    #"Qwen": [
        #LLMType.Qwen1B25,
    #    LLMType.Qwen72B25,
    #],
    #"GPT": [
        #LLMType.GPT3,
    #    LLMType.GPT4oMINI,
    #],
}

results = compute_answer_probability_tables_with_meta(
    dataset_name="CommonsenseQA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)


train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )
    
    prob_thre = 0.975
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_all.append(tables_by_llm_L2)
    end_L2 = time.time()
    print('L=2 finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_all.append(tables_by_llm_L3)
    end_L3 = time.time()
    print('L=3 finished! Time=',(end_L3-end_L2)/60,'mins')
    #tables_by_llm_L4 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 4)
    #tables_by_llm_L4_all.append(tables_by_llm_L4)
    #end_L4 = time.time()
    #print('L=4 finished! Time=',(end_L4-end_L3)/60,'mins')
    tables_by_llm_ACbeta = run_AC_beta_per_llm(test_results, conf_thre=prob_thre)
    tables_by_llm_ACbeta_all.append(tables_by_llm_ACbeta)

Num. of repeat: 0
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.01 mins
Finished 120/250 | Time spent: 0.01 mins
Finished 160/250 | Time spent: 0.01 mins
Finished 200/250 | Time spent: 0.02 mins
Finished 240/250 | Time spent: 0.02 mins
L=2 finished! Time= 0.01885996659596761 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.01 mins
Finished 160/250 | Time spent: 0.01 mins
Finished 200/250 | Time spent: 0.01 mins
Finished 240/250 | Time spent: 0.01 mins
L=3 finished! Time= 0.011255168914794922 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.01 mins
Finished 160/250 | Time spent: 0.01 mins
Finished 200/250 | Time spent: 0.01 mins
Finished 240/250 | Time spent: 0.01 mins
L=2 finished! Time= 0.009536449114481609 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/25

In [44]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_all, test_results, L_approx=3)

Prob. Threshold =  0.975
Average metrics per LLM (AC-Beta; rp=5):
LLaMA405B31           mean n_stop = 9.596    mode accuracy = 1.000    answer accuracy = 0.900

Average metrics per LLM (L=2; rp=5):
LLaMA405B31           mean n_stop = 7.717    mode accuracy = 0.990    answer accuracy = 0.893

Average metrics per LLM (L=3; rp=5):
LLaMA405B31           mean n_stop = 7.401    mode accuracy = 0.990    answer accuracy = 0.893



In [45]:
rp = 5
tables_by_llm_L2_all = []
tables_by_llm_L3_all = []
tables_by_llm_L4_all = []
tables_by_llm_ACbeta_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    #"Qwen": [
        #LLMType.Qwen1B25,
    #    LLMType.Qwen72B25,
    #],
    #"GPT": [
        #LLMType.GPT3,
    #    LLMType.GPT4oMINI,
    #],
}

results = compute_answer_probability_tables_with_meta(
    dataset_name="CommonsenseQA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)


train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )
    
    prob_thre = 0.95
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_all.append(tables_by_llm_L2)
    end_L2 = time.time()
    print('L=2 finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_all.append(tables_by_llm_L3)
    end_L3 = time.time()
    print('L=3 finished! Time=',(end_L3-end_L2)/60,'mins')
    #tables_by_llm_L4 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 4)
    #tables_by_llm_L4_all.append(tables_by_llm_L4)
    #end_L4 = time.time()
    #print('L=4 finished! Time=',(end_L4-end_L3)/60,'mins')
    tables_by_llm_ACbeta = run_AC_beta_per_llm(test_results, conf_thre=prob_thre)
    tables_by_llm_ACbeta_all.append(tables_by_llm_ACbeta)

Num. of repeat: 0
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.01 mins
Finished 120/250 | Time spent: 0.01 mins
Finished 160/250 | Time spent: 0.01 mins
Finished 200/250 | Time spent: 0.02 mins
Finished 240/250 | Time spent: 0.02 mins
L=2 finished! Time= 0.018247667948404947 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.01 mins
Finished 200/250 | Time spent: 0.01 mins
Finished 240/250 | Time spent: 0.01 mins
L=3 finished! Time= 0.009237198034922282 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.01 mins
Finished 200/250 | Time spent: 0.01 mins
Finished 240/250 | Time spent: 0.01 mins
L=2 finished! Time= 0.01044633388519287 mins
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.01 mins
Finished 120/25

In [46]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_all, test_results, L_approx=3)

Prob. Threshold =  0.95
Average metrics per LLM (AC-Beta; rp=5):
LLaMA405B31           mean n_stop = 7.548    mode accuracy = 0.996    answer accuracy = 0.904

Average metrics per LLM (L=2; rp=5):
LLaMA405B31           mean n_stop = 6.920    mode accuracy = 0.985    answer accuracy = 0.888

Average metrics per LLM (L=3; rp=5):
LLaMA405B31           mean n_stop = 6.497    mode accuracy = 0.983    answer accuracy = 0.886



In [25]:
rp = 5
tables_by_llm_L2_all = []
tables_by_llm_L3_all = []
tables_by_llm_L4_all = []
tables_by_llm_ACbeta_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ],
}

results = compute_answer_probability_tables_with_meta(
    dataset_name="AQUA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)


train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )
    
    prob_thre = 0.99
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_all.append(tables_by_llm_L2)
    end_L2 = time.time()
    print('L=2 finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_all.append(tables_by_llm_L3)
    end_L3 = time.time()
    print('L=3 finished! Time=',(end_L3-end_L2)/60,'mins')
    tables_by_llm_L4 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 4)
    tables_by_llm_L4_all.append(tables_by_llm_L4)
    end_L4 = time.time()
    print('L=4 finished! Time=',(end_L4-end_L3)/60,'mins')
    tables_by_llm_ACbeta = run_AC_beta_per_llm(test_results, conf_thre=prob_thre)
    tables_by_llm_ACbeta_all.append(tables_by_llm_ACbeta)
    


Num. of repeat: 0
Finished 40/77 | Time spent: 0.05 mins
L=2 finished! Time= 0.0681639035542806 mins
Finished 40/77 | Time spent: 0.13 mins
L=3 finished! Time= 0.17328114906946818 mins
Finished 40/77 | Time spent: 0.43 mins
L=4 finished! Time= 0.5271864811579386 mins
Num. of repeat: 1
Finished 40/77 | Time spent: 0.05 mins
L=2 finished! Time= 0.07187553246815999 mins
Finished 40/77 | Time spent: 0.13 mins
L=3 finished! Time= 0.16280116240183512 mins
Finished 40/77 | Time spent: 0.45 mins
L=4 finished! Time= 0.5081612348556519 mins
Num. of repeat: 2
Finished 40/77 | Time spent: 0.02 mins
L=2 finished! Time= 0.04832176764806111 mins
Finished 40/77 | Time spent: 0.03 mins
L=3 finished! Time= 0.06438260078430176 mins
Finished 40/77 | Time spent: 0.05 mins
L=4 finished! Time= 0.11974005301793417 mins
Num. of repeat: 3
Finished 40/77 | Time spent: 0.06 mins
L=2 finished! Time= 0.08679495255152385 mins
Finished 40/77 | Time spent: 0.15 mins
L=3 finished! Time= 0.20436456600824993 mins
Finishe

In [26]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_all, test_results, L_approx=3)
print_avg_metrics_rp(tables_by_llm_L4_all, test_results, L_approx=4)

Prob. Threshold =  0.99
Average metrics per LLM (AC-Beta; rp=5):
LLaMA405B31           mean n_stop = 12.000    mode accuracy = 1.000    answer accuracy = 0.870
Qwen72B25             mean n_stop = 9.545    mode accuracy = 1.000    answer accuracy = 0.870
GPT4oMINI             mean n_stop = 12.818    mode accuracy = 1.000    answer accuracy = 0.870

Average metrics per LLM (L=2; rp=5):
LLaMA405B31           mean n_stop = 11.714    mode accuracy = 0.984    answer accuracy = 0.875
Qwen72B25             mean n_stop = 9.797    mode accuracy = 0.992    answer accuracy = 0.873
GPT4oMINI             mean n_stop = 11.714    mode accuracy = 0.974    answer accuracy = 0.862

Average metrics per LLM (L=3; rp=5):
LLaMA405B31           mean n_stop = 11.182    mode accuracy = 0.984    answer accuracy = 0.875
Qwen72B25             mean n_stop = 9.545    mode accuracy = 0.992    answer accuracy = 0.873
GPT4oMINI             mean n_stop = 11.296    mode accuracy = 0.974    answer accuracy = 0.862

Averag

In [27]:
rp = 5
tables_by_llm_L2_all = []
tables_by_llm_L3_all = []
tables_by_llm_L4_all = []
tables_by_llm_ACbeta_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ],
}

results = compute_answer_probability_tables_with_meta(
    dataset_name="AQUA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)


train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )
    
    prob_thre = 0.975
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_all.append(tables_by_llm_L2)
    end_L2 = time.time()
    print('L=2 finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_all.append(tables_by_llm_L3)
    end_L3 = time.time()
    print('L=3 finished! Time=',(end_L3-end_L2)/60,'mins')
    tables_by_llm_L4 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 4)
    tables_by_llm_L4_all.append(tables_by_llm_L4)
    end_L4 = time.time()
    print('L=4 finished! Time=',(end_L4-end_L3)/60,'mins')
    tables_by_llm_ACbeta = run_AC_beta_per_llm(test_results, conf_thre=prob_thre)
    tables_by_llm_ACbeta_all.append(tables_by_llm_ACbeta)
    


Num. of repeat: 0
Finished 40/77 | Time spent: 0.05 mins
L=2 finished! Time= 0.06600653330485026 mins
Finished 40/77 | Time spent: 0.12 mins
L=3 finished! Time= 0.17057963609695434 mins
Finished 40/77 | Time spent: 0.41 mins
L=4 finished! Time= 0.5177538633346558 mins
Num. of repeat: 1
Finished 40/77 | Time spent: 0.05 mins
L=2 finished! Time= 0.061976730823516846 mins
Finished 40/77 | Time spent: 0.15 mins
L=3 finished! Time= 0.16275135278701783 mins
Finished 40/77 | Time spent: 0.52 mins
L=4 finished! Time= 0.5498506665229798 mins
Num. of repeat: 2
Finished 40/77 | Time spent: 0.02 mins
L=2 finished! Time= 0.0529365340868632 mins
Finished 40/77 | Time spent: 0.03 mins
L=3 finished! Time= 0.07166264851888021 mins
Finished 40/77 | Time spent: 0.05 mins
L=4 finished! Time= 0.13912766377131144 mins
Num. of repeat: 3
Finished 40/77 | Time spent: 0.07 mins
L=2 finished! Time= 0.1009172002474467 mins
Finished 40/77 | Time spent: 0.20 mins
L=3 finished! Time= 0.26078906853993733 mins
Finishe

In [28]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_all, test_results, L_approx=3)
print_avg_metrics_rp(tables_by_llm_L4_all, test_results, L_approx=4)

Prob. Threshold =  0.975
Average metrics per LLM (AC-Beta; rp=5):
LLaMA405B31           mean n_stop = 10.506    mode accuracy = 1.000    answer accuracy = 0.870
Qwen72B25             mean n_stop = 8.299    mode accuracy = 1.000    answer accuracy = 0.870
GPT4oMINI             mean n_stop = 11.000    mode accuracy = 1.000    answer accuracy = 0.870

Average metrics per LLM (L=2; rp=5):
LLaMA405B31           mean n_stop = 10.683    mode accuracy = 0.982    answer accuracy = 0.873
Qwen72B25             mean n_stop = 9.210    mode accuracy = 0.990    answer accuracy = 0.870
GPT4oMINI             mean n_stop = 10.501    mode accuracy = 0.974    answer accuracy = 0.862

Average metrics per LLM (L=3; rp=5):
LLaMA405B31           mean n_stop = 10.070    mode accuracy = 0.982    answer accuracy = 0.873
Qwen72B25             mean n_stop = 9.047    mode accuracy = 0.990    answer accuracy = 0.870
GPT4oMINI             mean n_stop = 10.221    mode accuracy = 0.974    answer accuracy = 0.862

Avera

In [29]:
rp = 5
tables_by_llm_L2_all = []
tables_by_llm_L3_all = []
tables_by_llm_L4_all = []
tables_by_llm_ACbeta_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ],
}

results = compute_answer_probability_tables_with_meta(
    dataset_name="AQUA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)


train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )
    
    prob_thre = 0.95
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_all.append(tables_by_llm_L2)
    end_L2 = time.time()
    print('L=2 finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_all.append(tables_by_llm_L3)
    end_L3 = time.time()
    print('L=3 finished! Time=',(end_L3-end_L2)/60,'mins')
    tables_by_llm_L4 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 4)
    tables_by_llm_L4_all.append(tables_by_llm_L4)
    end_L4 = time.time()
    print('L=4 finished! Time=',(end_L4-end_L3)/60,'mins')
    tables_by_llm_ACbeta = run_AC_beta_per_llm(test_results, conf_thre=prob_thre)
    tables_by_llm_ACbeta_all.append(tables_by_llm_ACbeta)

Num. of repeat: 0
Finished 40/77 | Time spent: 0.06 mins
L=2 finished! Time= 0.08080638249715169 mins
Finished 40/77 | Time spent: 0.13 mins
L=3 finished! Time= 0.1678062359491984 mins
Finished 40/77 | Time spent: 0.44 mins
L=4 finished! Time= 0.518575664361318 mins
Num. of repeat: 1
Finished 40/77 | Time spent: 0.05 mins
L=2 finished! Time= 0.06607917149861654 mins
Finished 40/77 | Time spent: 0.16 mins
L=3 finished! Time= 0.17977016766866047 mins
Finished 40/77 | Time spent: 0.56 mins
L=4 finished! Time= 0.5826317628224691 mins
Num. of repeat: 2
Finished 40/77 | Time spent: 0.02 mins
L=2 finished! Time= 0.051039032141367596 mins
Finished 40/77 | Time spent: 0.03 mins
L=3 finished! Time= 0.07454431851704915 mins
Finished 40/77 | Time spent: 0.05 mins
L=4 finished! Time= 0.13795861800511677 mins
Num. of repeat: 3
Finished 40/77 | Time spent: 0.06 mins
L=2 finished! Time= 0.09614607095718383 mins
Finished 40/77 | Time spent: 0.19 mins
L=3 finished! Time= 0.2489764451980591 mins
Finished

In [30]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_all, test_results, L_approx=3)
print_avg_metrics_rp(tables_by_llm_L4_all, test_results, L_approx=4)

Prob. Threshold =  0.95
Average metrics per LLM (AC-Beta; rp=5):
LLaMA405B31           mean n_stop = 7.935    mode accuracy = 1.000    answer accuracy = 0.870
Qwen72B25             mean n_stop = 7.143    mode accuracy = 1.000    answer accuracy = 0.870
GPT4oMINI             mean n_stop = 9.052    mode accuracy = 1.000    answer accuracy = 0.870

Average metrics per LLM (L=2; rp=5):
LLaMA405B31           mean n_stop = 9.470    mode accuracy = 0.974    answer accuracy = 0.868
Qwen72B25             mean n_stop = 8.548    mode accuracy = 0.990    answer accuracy = 0.870
GPT4oMINI             mean n_stop = 9.281    mode accuracy = 0.961    answer accuracy = 0.857

Average metrics per LLM (L=3; rp=5):
LLaMA405B31           mean n_stop = 9.195    mode accuracy = 0.974    answer accuracy = 0.868
Qwen72B25             mean n_stop = 8.462    mode accuracy = 0.990    answer accuracy = 0.870
GPT4oMINI             mean n_stop = 9.000    mode accuracy = 0.961    answer accuracy = 0.857

Average metr

In [36]:
rp = 5
tables_by_llm_L2_all = []
tables_by_llm_L3_all = []
tables_by_llm_L4_all = []
tables_by_llm_ACbeta_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ],
     "DeepSeek": [
        LLMType.DeepSeekV3,
    ],
}

results = compute_answer_probability_tables_with_meta(
    dataset_name="GSM8K",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)


train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )
    
    prob_thre = 0.99
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_all.append(tables_by_llm_L2)
    end_L2 = time.time()
    print('L=2 finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_all.append(tables_by_llm_L3)
    end_L3 = time.time()
    print('L=3 finished! Time=',(end_L3-end_L2)/60,'mins')
    #tables_by_llm_L4 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 4)
    #tables_by_llm_L4_all.append(tables_by_llm_L4)
    #end_L4 = time.time()
    #print('L=4 finished! Time=',(end_L4-end_L3)/60,'mins')
    tables_by_llm_ACbeta = run_AC_beta_per_llm(test_results, conf_thre=prob_thre)
    tables_by_llm_ACbeta_all.append(tables_by_llm_ACbeta)
    

Num. of repeat: 0
Finished 40/250 | Time spent: 0.44 mins
Finished 80/250 | Time spent: 0.54 mins
Finished 120/250 | Time spent: 0.97 mins
Finished 160/250 | Time spent: 1.50 mins
Finished 200/250 | Time spent: 1.94 mins
Finished 240/250 | Time spent: 2.08 mins
L=2 finished! Time= 2.0817904829978944 mins
Finished 40/250 | Time spent: 7.61 mins
Finished 80/250 | Time spent: 8.35 mins
Finished 120/250 | Time spent: 12.30 mins
Finished 160/250 | Time spent: 20.80 mins
Finished 200/250 | Time spent: 27.48 mins
Finished 240/250 | Time spent: 28.78 mins
L=3 finished! Time= 28.7796187321345 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.45 mins
Finished 80/250 | Time spent: 0.65 mins
Finished 120/250 | Time spent: 1.22 mins
Finished 160/250 | Time spent: 1.81 mins
Finished 200/250 | Time spent: 2.23 mins
Finished 240/250 | Time spent: 2.37 mins
L=2 finished! Time= 2.3778877019882203 mins
Finished 40/250 | Time spent: 7.53 mins
Finished 80/250 | Time spent: 9.76 mins
Finished 120/250 |

In [38]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_all, test_results, L_approx=3)


Prob. Threshold =  0.99
Average metrics per LLM (AC-Beta; rp=5):
LLaMA3B32             mean n_stop = 15.540    mode accuracy = 1.000    answer accuracy = 0.876
LLaMA405B31           mean n_stop = 7.016    mode accuracy = 1.000    answer accuracy = 0.972
Qwen72B25             mean n_stop = 6.908    mode accuracy = 1.000    answer accuracy = 0.972
GPT4oMINI             mean n_stop = 8.268    mode accuracy = 1.000    answer accuracy = 0.964
DeepSeekV3            mean n_stop = 6.740    mode accuracy = 1.000    answer accuracy = 0.968

Average metrics per LLM (L=2; rp=5):
LLaMA3B32             mean n_stop = 15.011    mode accuracy = 0.966    answer accuracy = 0.882
LLaMA405B31           mean n_stop = 2.311    mode accuracy = 1.000    answer accuracy = 0.972
Qwen72B25             mean n_stop = 2.506    mode accuracy = 0.998    answer accuracy = 0.972
GPT4oMINI             mean n_stop = 4.041    mode accuracy = 0.997    answer accuracy = 0.964
DeepSeekV3            mean n_stop = 2.286    mode

In [33]:
rp = 5
tables_by_llm_L2_all = []
tables_by_llm_L3_all = []
tables_by_llm_L4_all = []
tables_by_llm_ACbeta_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ],
     "DeepSeek": [
        LLMType.DeepSeekV3,
    ],
}

results = compute_answer_probability_tables_with_meta(
    dataset_name="GSM8K",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)


train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )
    
    prob_thre = 0.975
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_all.append(tables_by_llm_L2)
    end_L2 = time.time()
    print('L=2 finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_all.append(tables_by_llm_L3)
    end_L3 = time.time()
    print('L=3 finished! Time=',(end_L3-end_L2)/60,'mins')
    tables_by_llm_L4 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 4)
    tables_by_llm_L4_all.append(tables_by_llm_L4)
    end_L4 = time.time()
    print('L=4 finished! Time=',(end_L4-end_L3)/60,'mins')
    tables_by_llm_ACbeta = run_AC_beta_per_llm(test_results, conf_thre=prob_thre)
    tables_by_llm_ACbeta_all.append(tables_by_llm_ACbeta)

Num. of repeat: 0
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.01 mins
Finished 200/250 | Time spent: 0.01 mins
Finished 240/250 | Time spent: 0.01 mins
L=2 finished! Time= 0.014608414967854817 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.01 mins
Finished 200/250 | Time spent: 0.01 mins
Finished 240/250 | Time spent: 0.01 mins
L=3 finished! Time= 0.01252203385035197 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.02 mins
Finished 200/250 | Time spent: 0.02 mins
Finished 240/250 | Time spent: 0.02 mins
L=4 finished! Time= 0.01547556718190511 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250

In [34]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_all, test_results, L_approx=3)
print_avg_metrics_rp(tables_by_llm_L4_all, test_results, L_approx=4)

Prob. Threshold =  0.975
Average metrics per LLM (AC-Beta; rp=5):
LLaMA405B31           mean n_stop = 5.776    mode accuracy = 1.000    answer accuracy = 0.972
Qwen72B25             mean n_stop = 5.608    mode accuracy = 1.000    answer accuracy = 0.972
GPT4oMINI             mean n_stop = 6.644    mode accuracy = 0.996    answer accuracy = 0.968
DeepSeekV3            mean n_stop = 5.672    mode accuracy = 1.000    answer accuracy = 0.968

Average metrics per LLM (L=2; rp=5):
LLaMA405B31           mean n_stop = 2.085    mode accuracy = 0.998    answer accuracy = 0.970
Qwen72B25             mean n_stop = 2.186    mode accuracy = 0.998    answer accuracy = 0.972
GPT4oMINI             mean n_stop = 3.810    mode accuracy = 0.994    answer accuracy = 0.966
DeepSeekV3            mean n_stop = 1.974    mode accuracy = 1.000    answer accuracy = 0.968

Average metrics per LLM (L=3; rp=5):
LLaMA405B31           mean n_stop = 2.043    mode accuracy = 0.999    answer accuracy = 0.971
Qwen72B25   

In [39]:
rp = 5
tables_by_llm_L2_all = []
tables_by_llm_L3_all = []
tables_by_llm_L4_all = []
tables_by_llm_ACbeta_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ],
     "DeepSeek": [
        LLMType.DeepSeekV3,
    ],
}

results = compute_answer_probability_tables_with_meta(
    dataset_name="GSM8K",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)


train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )
    
    prob_thre = 0.95
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_all.append(tables_by_llm_L2)
    end_L2 = time.time()
    print('L=2 finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_all.append(tables_by_llm_L3)
    end_L3 = time.time()
    print('L=3 finished! Time=',(end_L3-end_L2)/60,'mins')
    tables_by_llm_L4 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 4)
    tables_by_llm_L4_all.append(tables_by_llm_L4)
    end_L4 = time.time()
    print('L=4 finished! Time=',(end_L4-end_L3)/60,'mins')
    tables_by_llm_ACbeta = run_AC_beta_per_llm(test_results, conf_thre=prob_thre)
    tables_by_llm_ACbeta_all.append(tables_by_llm_ACbeta)

Num. of repeat: 0
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.01 mins
Finished 200/250 | Time spent: 0.01 mins
Finished 240/250 | Time spent: 0.01 mins
L=2 finished! Time= 0.012493634223937988 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.01 mins
Finished 200/250 | Time spent: 0.01 mins
Finished 240/250 | Time spent: 0.01 mins
L=3 finished! Time= 0.010878535111745198 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.01 mins
Finished 200/250 | Time spent: 0.01 mins
Finished 240/250 | Time spent: 0.01 mins
L=4 finished! Time= 0.013312649726867676 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/2

In [40]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_all, test_results, L_approx=3)
print_avg_metrics_rp(tables_by_llm_L4_all, test_results, L_approx=4)

Prob. Threshold =  0.95
Average metrics per LLM (AC-Beta; rp=5):
LLaMA405B31           mean n_stop = 4.496    mode accuracy = 1.000    answer accuracy = 0.972
Qwen72B25             mean n_stop = 4.576    mode accuracy = 1.000    answer accuracy = 0.972
GPT4oMINI             mean n_stop = 5.328    mode accuracy = 0.996    answer accuracy = 0.968
DeepSeekV3            mean n_stop = 4.612    mode accuracy = 1.000    answer accuracy = 0.968

Average metrics per LLM (L=2; rp=5):
LLaMA405B31           mean n_stop = 1.947    mode accuracy = 0.998    answer accuracy = 0.970
Qwen72B25             mean n_stop = 2.033    mode accuracy = 0.998    answer accuracy = 0.971
GPT4oMINI             mean n_stop = 3.527    mode accuracy = 0.994    answer accuracy = 0.966
DeepSeekV3            mean n_stop = 1.843    mode accuracy = 0.998    answer accuracy = 0.966

Average metrics per LLM (L=3; rp=5):
LLaMA405B31           mean n_stop = 1.881    mode accuracy = 0.998    answer accuracy = 0.970
Qwen72B25    

In [52]:
rp = 5
tables_by_llm_L2_all = []
tables_by_llm_L3_all = []
tables_by_llm_L4_all = []
tables_by_llm_ACbeta_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ],
}

results = compute_answer_probability_tables_with_meta(
    dataset_name="BIGBENCH_DisambiguationQA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)


train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )
    
    prob_thre = 0.99
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_all.append(tables_by_llm_L2)
    end_L2 = time.time()
    print('L=2 finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_all.append(tables_by_llm_L3)
    end_L3 = time.time()
    print('L=3 finished! Time=',(end_L3-end_L2)/60,'mins')
    #tables_by_llm_L4 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 4)
    #tables_by_llm_L4_all.append(tables_by_llm_L4)
    #end_L4 = time.time()
    #print('L=4 finished! Time=',(end_L4-end_L3)/60,'mins')
    tables_by_llm_ACbeta = run_AC_beta_per_llm(test_results, conf_thre=prob_thre)
    tables_by_llm_ACbeta_all.append(tables_by_llm_ACbeta)

Num. of repeat: 0
Finished 40/75 | Time spent: 0.00 mins
L=2 finished! Time= 0.0005388498306274414 mins
Finished 40/75 | Time spent: 0.00 mins
L=3 finished! Time= 0.000293115774790446 mins
Num. of repeat: 1
Finished 40/75 | Time spent: 0.00 mins
L=2 finished! Time= 0.0004784146944681803 mins
Finished 40/75 | Time spent: 0.00 mins
L=3 finished! Time= 0.00030464728673299153 mins
Num. of repeat: 2
Finished 40/75 | Time spent: 0.00 mins
L=2 finished! Time= 0.0007289687792460124 mins
Finished 40/75 | Time spent: 0.00 mins
L=3 finished! Time= 0.0003632783889770508 mins
Num. of repeat: 3
Finished 40/75 | Time spent: 0.00 mins
L=2 finished! Time= 0.0005044341087341309 mins
Finished 40/75 | Time spent: 0.00 mins
L=3 finished! Time= 0.00036386648813883465 mins
Num. of repeat: 4
Finished 40/75 | Time spent: 0.00 mins
L=2 finished! Time= 0.0004317204157511393 mins
Finished 40/75 | Time spent: 0.00 mins
L=3 finished! Time= 0.00028363466262817385 mins


In [53]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_all, test_results, L_approx=3)

Prob. Threshold =  0.99
Average metrics per LLM (AC-Beta; rp=5):
LLaMA405B31           mean n_stop = 11.827    mode accuracy = 1.000    answer accuracy = 0.760
Qwen72B25             mean n_stop = 9.640    mode accuracy = 1.000    answer accuracy = 0.880
GPT4oMINI             mean n_stop = 12.560    mode accuracy = 1.000    answer accuracy = 0.787

Average metrics per LLM (L=2; rp=5):
LLaMA405B31           mean n_stop = 5.677    mode accuracy = 0.992    answer accuracy = 0.763
Qwen72B25             mean n_stop = 6.107    mode accuracy = 0.984    answer accuracy = 0.891
GPT4oMINI             mean n_stop = 10.549    mode accuracy = 0.995    answer accuracy = 0.781

Average metrics per LLM (L=3; rp=5):
LLaMA405B31           mean n_stop = 5.627    mode accuracy = 0.992    answer accuracy = 0.763
Qwen72B25             mean n_stop = 6.088    mode accuracy = 0.984    answer accuracy = 0.891
GPT4oMINI             mean n_stop = 10.549    mode accuracy = 0.995    answer accuracy = 0.781



In [54]:
rp = 5
tables_by_llm_L2_all = []
tables_by_llm_L3_all = []
tables_by_llm_L4_all = []
tables_by_llm_ACbeta_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ],
}

results = compute_answer_probability_tables_with_meta(
    dataset_name="BIGBENCH_DisambiguationQA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)


train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )
    
    prob_thre = 0.975
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_all.append(tables_by_llm_L2)
    end_L2 = time.time()
    print('L=2 finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_all.append(tables_by_llm_L3)
    end_L3 = time.time()
    print('L=3 finished! Time=',(end_L3-end_L2)/60,'mins')
    #tables_by_llm_L4 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 4)
    #tables_by_llm_L4_all.append(tables_by_llm_L4)
    #end_L4 = time.time()
    #print('L=4 finished! Time=',(end_L4-end_L3)/60,'mins')
    tables_by_llm_ACbeta = run_AC_beta_per_llm(test_results, conf_thre=prob_thre)
    tables_by_llm_ACbeta_all.append(tables_by_llm_ACbeta)

Num. of repeat: 0
Finished 40/75 | Time spent: 0.00 mins
L=2 finished! Time= 0.00046138366063435874 mins
Finished 40/75 | Time spent: 0.00 mins
L=3 finished! Time= 0.00025535027186075846 mins
Num. of repeat: 1
Finished 40/75 | Time spent: 0.00 mins
L=2 finished! Time= 0.0004018704096476237 mins
Finished 40/75 | Time spent: 0.00 mins
L=3 finished! Time= 0.00026057958602905275 mins
Num. of repeat: 2
Finished 40/75 | Time spent: 0.00 mins
L=2 finished! Time= 0.0006058335304260254 mins
Finished 40/75 | Time spent: 0.00 mins
L=3 finished! Time= 0.0003003994623819987 mins
Num. of repeat: 3
Finished 40/75 | Time spent: 0.00 mins
L=2 finished! Time= 0.0004095633824666341 mins
Finished 40/75 | Time spent: 0.00 mins
L=3 finished! Time= 0.0002725680669148763 mins
Num. of repeat: 4
Finished 40/75 | Time spent: 0.00 mins
L=2 finished! Time= 0.0003523667653401693 mins
Finished 40/75 | Time spent: 0.00 mins
L=3 finished! Time= 0.00022563536961873372 mins


In [55]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_all, test_results, L_approx=3)

Prob. Threshold =  0.975
Average metrics per LLM (AC-Beta; rp=5):
LLaMA405B31           mean n_stop = 9.493    mode accuracy = 1.000    answer accuracy = 0.760
Qwen72B25             mean n_stop = 7.667    mode accuracy = 1.000    answer accuracy = 0.880
GPT4oMINI             mean n_stop = 10.240    mode accuracy = 1.000    answer accuracy = 0.787

Average metrics per LLM (L=2; rp=5):
LLaMA405B31           mean n_stop = 4.781    mode accuracy = 0.992    answer accuracy = 0.763
Qwen72B25             mean n_stop = 5.536    mode accuracy = 0.984    answer accuracy = 0.891
GPT4oMINI             mean n_stop = 8.904    mode accuracy = 0.995    answer accuracy = 0.781

Average metrics per LLM (L=3; rp=5):
LLaMA405B31           mean n_stop = 4.739    mode accuracy = 0.992    answer accuracy = 0.763
Qwen72B25             mean n_stop = 5.483    mode accuracy = 0.984    answer accuracy = 0.891
GPT4oMINI             mean n_stop = 8.904    mode accuracy = 0.995    answer accuracy = 0.781



In [56]:
rp = 5
tables_by_llm_L2_all = []
tables_by_llm_L3_all = []
tables_by_llm_L4_all = []
tables_by_llm_ACbeta_all = []
model_classes = {
    "LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        LLMType.LLaMA405B31,
    ],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        LLMType.GPT4oMINI,
    ],
}

results = compute_answer_probability_tables_with_meta(
    dataset_name="BIGBENCH_DisambiguationQA",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)


train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )
    
    prob_thre = 0.95
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_all.append(tables_by_llm_L2)
    end_L2 = time.time()
    print('L=2 finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_all.append(tables_by_llm_L3)
    end_L3 = time.time()
    print('L=3 finished! Time=',(end_L3-end_L2)/60,'mins')
    #tables_by_llm_L4 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 4)
    #tables_by_llm_L4_all.append(tables_by_llm_L4)
    #end_L4 = time.time()
    #print('L=4 finished! Time=',(end_L4-end_L3)/60,'mins')
    tables_by_llm_ACbeta = run_AC_beta_per_llm(test_results, conf_thre=prob_thre)
    tables_by_llm_ACbeta_all.append(tables_by_llm_ACbeta)

Num. of repeat: 0
Finished 40/75 | Time spent: 0.00 mins
L=2 finished! Time= 0.0003943324089050293 mins
Finished 40/75 | Time spent: 0.00 mins
L=3 finished! Time= 0.00020534992218017577 mins
Num. of repeat: 1
Finished 40/75 | Time spent: 0.00 mins
L=2 finished! Time= 0.000334318478902181 mins
Finished 40/75 | Time spent: 0.00 mins
L=3 finished! Time= 0.0002213875452677409 mins
Num. of repeat: 2
Finished 40/75 | Time spent: 0.00 mins
L=2 finished! Time= 0.0005506833394368489 mins
Finished 40/75 | Time spent: 0.00 mins
L=3 finished! Time= 0.0002531846364339193 mins
Num. of repeat: 3
Finished 40/75 | Time spent: 0.00 mins
L=2 finished! Time= 0.000370331605275472 mins
Finished 40/75 | Time spent: 0.00 mins
L=3 finished! Time= 0.00022914807001749675 mins
Num. of repeat: 4
Finished 40/75 | Time spent: 0.00 mins
L=2 finished! Time= 0.0002605120340983073 mins
Finished 40/75 | Time spent: 0.00 mins
L=3 finished! Time= 0.00019455353418986002 mins


In [57]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_all, test_results, L_approx=3)

Prob. Threshold =  0.95
Average metrics per LLM (AC-Beta; rp=5):
LLaMA405B31           mean n_stop = 7.493    mode accuracy = 1.000    answer accuracy = 0.760
Qwen72B25             mean n_stop = 6.187    mode accuracy = 1.000    answer accuracy = 0.880
GPT4oMINI             mean n_stop = 8.053    mode accuracy = 1.000    answer accuracy = 0.787

Average metrics per LLM (L=2; rp=5):
LLaMA405B31           mean n_stop = 3.867    mode accuracy = 0.976    answer accuracy = 0.757
Qwen72B25             mean n_stop = 4.933    mode accuracy = 0.979    answer accuracy = 0.885
GPT4oMINI             mean n_stop = 7.645    mode accuracy = 0.987    answer accuracy = 0.779

Average metrics per LLM (L=3; rp=5):
LLaMA405B31           mean n_stop = 3.600    mode accuracy = 0.973    answer accuracy = 0.760
Qwen72B25             mean n_stop = 4.928    mode accuracy = 0.979    answer accuracy = 0.885
GPT4oMINI             mean n_stop = 7.645    mode accuracy = 0.987    answer accuracy = 0.779



In [10]:
rp = 20
tables_by_llm_L2_all = []
tables_by_llm_L3_all = []
tables_by_llm_L4_all = []
tables_by_llm_ACbeta_all = []
model_classes = {
    #"LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        #LLMType.LLaMA405B31,
    #],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        #LLMType.GPT4oMINI,
    ],
}

results = compute_answer_probability_tables_with_meta(
    dataset_name="GSM8K",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)


train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )
    
    prob_thre = 0.99
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_all.append(tables_by_llm_L2)
    end_L2 = time.time()
    print('L=2 finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_all.append(tables_by_llm_L3)
    end_L3 = time.time()
    print('L=3 finished! Time=',(end_L3-end_L2)/60,'mins')
    tables_by_llm_L4 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 4)
    tables_by_llm_L4_all.append(tables_by_llm_L4)
    end_L4 = time.time()
    print('L=4 finished! Time=',(end_L4-end_L3)/60,'mins')
    tables_by_llm_ACbeta = run_AC_beta_per_llm(test_results, conf_thre=prob_thre)
    tables_by_llm_ACbeta_all.append(tables_by_llm_ACbeta)
    


Num. of repeat: 0
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=2 finished! Time= 0.004364049434661866 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.01 mins
Finished 200/250 | Time spent: 0.01 mins
Finished 240/250 | Time spent: 0.01 mins
L=3 finished! Time= 0.0138006329536438 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.05 mins
Finished 200/250 | Time spent: 0.05 mins
Finished 240/250 | Time spent: 0.05 mins
L=4 finished! Time= 0.051824867725372314 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250

Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=4 finished! Time= 0.004730486869812011 mins
Num. of repeat: 10
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=2 finished! Time= 0.0016678651173909506 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=3 finished! Time= 0.003134620189666748 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.01 mins
Finished 200/250 | Time spent: 0.01 mins
Finished 240/250 | Time spen

Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=3 finished! Time= 0.0024631182352701825 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.01 mins
Finished 240/250 | Time spent: 0.01 mins
L=4 finished! Time= 0.005592314402262369 mins


In [11]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_all, test_results, L_approx=3)

Prob. Threshold =  0.99
Average metrics per LLM (AC-Beta; rp=20):
Qwen72B25             mean n_stop = 6.908 ± 0.135    mode accuracy = 1.000 ± 0.000    answer accuracy = 0.972 ± 0.005

Average metrics per LLM (L=2; rp=20):
Qwen72B25             mean n_stop = 2.473 ± 0.271    mode accuracy = 0.998 ± 0.001    answer accuracy = 0.972 ± 0.005

Average metrics per LLM (L=3; rp=20):
Qwen72B25             mean n_stop = 2.429 ± 0.266    mode accuracy = 0.998 ± 0.001    answer accuracy = 0.972 ± 0.005



In [12]:
rp = 20
tables_by_llm_L2_all = []
tables_by_llm_L3_all = []
tables_by_llm_L4_all = []
tables_by_llm_ACbeta_all = []
model_classes = {
    #"LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        #LLMType.LLaMA405B31,
    #],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        #LLMType.GPT4oMINI,
    ],
}

results = compute_answer_probability_tables_with_meta(
    dataset_name="GSM8K",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)


train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )
    
    prob_thre = 0.975
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_all.append(tables_by_llm_L2)
    end_L2 = time.time()
    print('L=2 finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_all.append(tables_by_llm_L3)
    end_L3 = time.time()
    print('L=3 finished! Time=',(end_L3-end_L2)/60,'mins')
    tables_by_llm_L4 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 4)
    tables_by_llm_L4_all.append(tables_by_llm_L4)
    end_L4 = time.time()
    print('L=4 finished! Time=',(end_L4-end_L3)/60,'mins')
    tables_by_llm_ACbeta = run_AC_beta_per_llm(test_results, conf_thre=prob_thre)
    tables_by_llm_ACbeta_all.append(tables_by_llm_ACbeta)
    

Num. of repeat: 0
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=2 finished! Time= 0.003428204854329427 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.01 mins
Finished 200/250 | Time spent: 0.01 mins
Finished 240/250 | Time spent: 0.01 mins
L=3 finished! Time= 0.00977557897567749 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.04 mins
Finished 200/250 | Time spent: 0.04 mins
Finished 240/250 | Time spent: 0.04 mins
L=4 finished! Time= 0.041320053736368816 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/25

Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=4 finished! Time= 0.0036495010058085124 mins
Num. of repeat: 10
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=2 finished! Time= 0.0009099165598551432 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=3 finished! Time= 0.0026498993237813314 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.01 mins
Finished 200/250 | Time spent: 0.01 mins
Finished 240/250 | Time sp

Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=3 finished! Time= 0.001941680908203125 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=4 finished! Time= 0.004487832387288411 mins


In [13]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_all, test_results, L_approx=3)

Prob. Threshold =  0.975
Average metrics per LLM (AC-Beta; rp=20):
Qwen72B25             mean n_stop = 5.608 ± 0.104    mode accuracy = 1.000 ± 0.000    answer accuracy = 0.972 ± 0.005

Average metrics per LLM (L=2; rp=20):
Qwen72B25             mean n_stop = 2.321 ± 0.258    mode accuracy = 0.998 ± 0.001    answer accuracy = 0.972 ± 0.005

Average metrics per LLM (L=3; rp=20):
Qwen72B25             mean n_stop = 2.288 ± 0.255    mode accuracy = 0.998 ± 0.001    answer accuracy = 0.972 ± 0.005



In [14]:
rp = 20
tables_by_llm_L2_all = []
tables_by_llm_L3_all = []
tables_by_llm_L4_all = []
tables_by_llm_ACbeta_all = []
model_classes = {
    #"LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        #LLMType.LLaMA405B31,
    #],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        #LLMType.GPT4oMINI,
    ],
}

results = compute_answer_probability_tables_with_meta(
    dataset_name="GSM8K",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)


train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )
    
    prob_thre = 0.95
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_all.append(tables_by_llm_L2)
    end_L2 = time.time()
    print('L=2 finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_all.append(tables_by_llm_L3)
    end_L3 = time.time()
    print('L=3 finished! Time=',(end_L3-end_L2)/60,'mins')
    tables_by_llm_L4 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 4)
    tables_by_llm_L4_all.append(tables_by_llm_L4)
    end_L4 = time.time()
    print('L=4 finished! Time=',(end_L4-end_L3)/60,'mins')
    tables_by_llm_ACbeta = run_AC_beta_per_llm(test_results, conf_thre=prob_thre)
    tables_by_llm_ACbeta_all.append(tables_by_llm_ACbeta)
    

Num. of repeat: 0
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=2 finished! Time= 0.0032056331634521484 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=3 finished! Time= 0.0034686525662740073 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.01 mins
Finished 200/250 | Time spent: 0.02 mins
Finished 240/250 | Time spent: 0.02 mins
L=4 finished! Time= 0.016189384460449218 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120

Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=4 finished! Time= 0.003055286407470703 mins
Num. of repeat: 10
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=2 finished! Time= 0.000748598575592041 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=3 finished! Time= 0.002493902047475179 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.01 mins
Finished 200/250 | Time spent: 0.01 mins
Finished 240/250 | Time spent

Finished 80/250 | Time spent: 0.00 mins
Finished 120/250 | Time spent: 0.00 mins
Finished 160/250 | Time spent: 0.00 mins
Finished 200/250 | Time spent: 0.00 mins
Finished 240/250 | Time spent: 0.00 mins
L=4 finished! Time= 0.0034707307815551756 mins


In [15]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_all, test_results, L_approx=3)

Prob. Threshold =  0.95
Average metrics per LLM (AC-Beta; rp=20):
Qwen72B25             mean n_stop = 4.576 ± 0.103    mode accuracy = 1.000 ± 0.000    answer accuracy = 0.972 ± 0.005

Average metrics per LLM (L=2; rp=20):
Qwen72B25             mean n_stop = 2.178 ± 0.247    mode accuracy = 0.997 ± 0.002    answer accuracy = 0.970 ± 0.005

Average metrics per LLM (L=3; rp=20):
Qwen72B25             mean n_stop = 2.132 ± 0.242    mode accuracy = 0.997 ± 0.002    answer accuracy = 0.970 ± 0.005



In [16]:
rp = 20
tables_by_llm_L2_all = []
tables_by_llm_L3_all = []
tables_by_llm_L4_all = []
tables_by_llm_ACbeta_all = []
model_classes = {
    #"LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        #LLMType.LLaMA405B31,
    #],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        #LLMType.GPT4oMINI,
    ],
}

results = compute_answer_probability_tables_with_meta(
    dataset_name="SVAMP",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)


train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )
    
    prob_thre = 0.99
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_all.append(tables_by_llm_L2)
    end_L2 = time.time()
    print('L=2 finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_all.append(tables_by_llm_L3)
    end_L3 = time.time()
    print('L=3 finished! Time=',(end_L3-end_L2)/60,'mins')
    tables_by_llm_L4 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 4)
    tables_by_llm_L4_all.append(tables_by_llm_L4)
    end_L4 = time.time()
    print('L=4 finished! Time=',(end_L4-end_L3)/60,'mins')
    tables_by_llm_ACbeta = run_AC_beta_per_llm(test_results, conf_thre=prob_thre)
    tables_by_llm_ACbeta_all.append(tables_by_llm_ACbeta)
    

Num. of repeat: 0
Finished 40/250 | Time spent: 0.02 mins
Finished 80/250 | Time spent: 0.04 mins
Finished 120/250 | Time spent: 0.04 mins
Finished 160/250 | Time spent: 0.04 mins
Finished 200/250 | Time spent: 0.04 mins
Finished 240/250 | Time spent: 0.04 mins
L=2 finished! Time= 0.044380935033162434 mins
Finished 40/250 | Time spent: 0.10 mins
Finished 80/250 | Time spent: 0.15 mins
Finished 120/250 | Time spent: 0.15 mins
Finished 160/250 | Time spent: 0.15 mins
Finished 200/250 | Time spent: 0.15 mins
Finished 240/250 | Time spent: 0.15 mins
L=3 finished! Time= 0.15488796234130858 mins
Finished 40/250 | Time spent: 0.50 mins
Finished 80/250 | Time spent: 0.79 mins
Finished 120/250 | Time spent: 0.79 mins
Finished 160/250 | Time spent: 0.79 mins
Finished 200/250 | Time spent: 0.79 mins
Finished 240/250 | Time spent: 0.79 mins
L=4 finished! Time= 0.7877235849698384 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.03 mins
Finished 80/250 | Time spent: 0.04 mins
Finished 120/250 

Finished 40/250 | Time spent: 0.05 mins
Finished 80/250 | Time spent: 0.06 mins
Finished 120/250 | Time spent: 0.06 mins
Finished 160/250 | Time spent: 0.06 mins
Finished 200/250 | Time spent: 0.06 mins
Finished 240/250 | Time spent: 0.06 mins
L=3 finished! Time= 0.05687646865844727 mins
Finished 40/250 | Time spent: 0.27 mins
Finished 80/250 | Time spent: 0.29 mins
Finished 120/250 | Time spent: 0.29 mins
Finished 160/250 | Time spent: 0.29 mins
Finished 200/250 | Time spent: 0.29 mins
Finished 240/250 | Time spent: 0.29 mins
L=4 finished! Time= 0.28584221601486204 mins
Num. of repeat: 10
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.01 mins
Finished 120/250 | Time spent: 0.01 mins
Finished 160/250 | Time spent: 0.01 mins
Finished 200/250 | Time spent: 0.01 mins
Finished 240/250 | Time spent: 0.01 mins
L=2 finished! Time= 0.012811052799224853 mins
Finished 40/250 | Time spent: 0.03 mins
Finished 80/250 | Time spent: 0.03 mins
Finished 120/250 | Time spent: 0.

Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.02 mins
Finished 120/250 | Time spent: 0.02 mins
Finished 160/250 | Time spent: 0.02 mins
Finished 200/250 | Time spent: 0.02 mins
Finished 240/250 | Time spent: 0.02 mins
L=4 finished! Time= 0.023576116561889647 mins
Num. of repeat: 19
Finished 40/250 | Time spent: 0.04 mins
Finished 80/250 | Time spent: 0.05 mins
Finished 120/250 | Time spent: 0.05 mins
Finished 160/250 | Time spent: 0.05 mins
Finished 200/250 | Time spent: 0.05 mins
Finished 240/250 | Time spent: 0.05 mins
L=2 finished! Time= 0.04937761624654134 mins
Finished 40/250 | Time spent: 0.13 mins
Finished 80/250 | Time spent: 0.14 mins
Finished 120/250 | Time spent: 0.14 mins
Finished 160/250 | Time spent: 0.14 mins
Finished 200/250 | Time spent: 0.14 mins
Finished 240/250 | Time spent: 0.14 mins
L=3 finished! Time= 0.13864656686782836 mins
Finished 40/250 | Time spent: 0.74 mins
Finished 80/250 | Time spent: 0.76 mins
Finished 120/250 | Time spent: 0.

In [17]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_all, test_results, L_approx=3)

Prob. Threshold =  0.99
Average metrics per LLM (AC-Beta; rp=20):
Qwen72B25             mean n_stop = 7.140 ± 0.150    mode accuracy = 1.000 ± 0.000    answer accuracy = 0.952 ± 0.006

Average metrics per LLM (L=2; rp=20):
Qwen72B25             mean n_stop = 2.914 ± 0.315    mode accuracy = 0.997 ± 0.001    answer accuracy = 0.950 ± 0.006

Average metrics per LLM (L=3; rp=20):
Qwen72B25             mean n_stop = 2.793 ± 0.300    mode accuracy = 0.997 ± 0.001    answer accuracy = 0.950 ± 0.006



In [18]:
rp = 20
tables_by_llm_L2_all = []
tables_by_llm_L3_all = []
tables_by_llm_L4_all = []
tables_by_llm_ACbeta_all = []
model_classes = {
    #"LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        #LLMType.LLaMA405B31,
    #],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        #LLMType.GPT4oMINI,
    ],
}

results = compute_answer_probability_tables_with_meta(
    dataset_name="SVAMP",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)


train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )
    
    prob_thre = 0.975
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_all.append(tables_by_llm_L2)
    end_L2 = time.time()
    print('L=2 finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_all.append(tables_by_llm_L3)
    end_L3 = time.time()
    print('L=3 finished! Time=',(end_L3-end_L2)/60,'mins')
    tables_by_llm_L4 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 4)
    tables_by_llm_L4_all.append(tables_by_llm_L4)
    end_L4 = time.time()
    print('L=4 finished! Time=',(end_L4-end_L3)/60,'mins')
    tables_by_llm_ACbeta = run_AC_beta_per_llm(test_results, conf_thre=prob_thre)
    tables_by_llm_ACbeta_all.append(tables_by_llm_ACbeta)
    

Num. of repeat: 0
Finished 40/250 | Time spent: 0.02 mins
Finished 80/250 | Time spent: 0.04 mins
Finished 120/250 | Time spent: 0.04 mins
Finished 160/250 | Time spent: 0.04 mins
Finished 200/250 | Time spent: 0.04 mins
Finished 240/250 | Time spent: 0.04 mins
L=2 finished! Time= 0.04074045022328695 mins
Finished 40/250 | Time spent: 0.09 mins
Finished 80/250 | Time spent: 0.15 mins
Finished 120/250 | Time spent: 0.15 mins
Finished 160/250 | Time spent: 0.15 mins
Finished 200/250 | Time spent: 0.15 mins
Finished 240/250 | Time spent: 0.15 mins
L=3 finished! Time= 0.15168963273366293 mins
Finished 40/250 | Time spent: 0.48 mins
Finished 80/250 | Time spent: 0.77 mins
Finished 120/250 | Time spent: 0.77 mins
Finished 160/250 | Time spent: 0.77 mins
Finished 200/250 | Time spent: 0.77 mins
Finished 240/250 | Time spent: 0.77 mins
L=4 finished! Time= 0.7711679140726725 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.02 mins
Finished 80/250 | Time spent: 0.03 mins
Finished 120/250 |

Finished 40/250 | Time spent: 0.05 mins
Finished 80/250 | Time spent: 0.05 mins
Finished 120/250 | Time spent: 0.05 mins
Finished 160/250 | Time spent: 0.05 mins
Finished 200/250 | Time spent: 0.05 mins
Finished 240/250 | Time spent: 0.05 mins
L=3 finished! Time= 0.0502843181292216 mins
Finished 40/250 | Time spent: 0.26 mins
Finished 80/250 | Time spent: 0.27 mins
Finished 120/250 | Time spent: 0.27 mins
Finished 160/250 | Time spent: 0.27 mins
Finished 200/250 | Time spent: 0.27 mins
Finished 240/250 | Time spent: 0.27 mins
L=4 finished! Time= 0.2667329668998718 mins
Num. of repeat: 10
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.01 mins
Finished 120/250 | Time spent: 0.01 mins
Finished 160/250 | Time spent: 0.01 mins
Finished 200/250 | Time spent: 0.01 mins
Finished 240/250 | Time spent: 0.01 mins
L=2 finished! Time= 0.012006449699401855 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.01 mins
Finished 120/250 | Time spent: 0.01

Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.02 mins
Finished 120/250 | Time spent: 0.02 mins
Finished 160/250 | Time spent: 0.02 mins
Finished 200/250 | Time spent: 0.02 mins
Finished 240/250 | Time spent: 0.02 mins
L=4 finished! Time= 0.019003566106160483 mins
Num. of repeat: 19
Finished 40/250 | Time spent: 0.02 mins
Finished 80/250 | Time spent: 0.03 mins
Finished 120/250 | Time spent: 0.03 mins
Finished 160/250 | Time spent: 0.03 mins
Finished 200/250 | Time spent: 0.03 mins
Finished 240/250 | Time spent: 0.03 mins
L=2 finished! Time= 0.034238898754119874 mins
Finished 40/250 | Time spent: 0.08 mins
Finished 80/250 | Time spent: 0.08 mins
Finished 120/250 | Time spent: 0.08 mins
Finished 160/250 | Time spent: 0.08 mins
Finished 200/250 | Time spent: 0.09 mins
Finished 240/250 | Time spent: 0.09 mins
L=3 finished! Time= 0.08534988562266031 mins
Finished 40/250 | Time spent: 0.46 mins
Finished 80/250 | Time spent: 0.47 mins
Finished 120/250 | Time spent: 0

In [19]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_all, test_results, L_approx=3)

Prob. Threshold =  0.975
Average metrics per LLM (AC-Beta; rp=20):
Qwen72B25             mean n_stop = 5.892 ± 0.132    mode accuracy = 1.000 ± 0.000    answer accuracy = 0.952 ± 0.006

Average metrics per LLM (L=2; rp=20):
Qwen72B25             mean n_stop = 2.695 ± 0.296    mode accuracy = 0.997 ± 0.002    answer accuracy = 0.950 ± 0.006

Average metrics per LLM (L=3; rp=20):
Qwen72B25             mean n_stop = 2.565 ± 0.280    mode accuracy = 0.997 ± 0.002    answer accuracy = 0.950 ± 0.006



In [20]:
rp = 20
tables_by_llm_L2_all = []
tables_by_llm_L3_all = []
tables_by_llm_L4_all = []
tables_by_llm_ACbeta_all = []
model_classes = {
    #"LLaMA": [
        #LLMType.LLaMA1B32,
        #LLMType.LLaMA3B32,
        #LLMType.LLaMA8B31,
        #LLMType.LLaMA405B31,
    #],
    "Qwen": [
        #LLMType.Qwen1B25,
        LLMType.Qwen72B25,
    ],
    "GPT": [
        #LLMType.GPT3,
        #LLMType.GPT4oMINI,
    ],
}

results = compute_answer_probability_tables_with_meta(
    dataset_name="SVAMP",
    model_classes=model_classes,
    api_path=api_path_global,
    N_samples=40,
)


train_results, test_results = split_results_train_test_limited(results, train_ratio=0.7, random_state=42)


for rp_idx in range(rp):
    new_test_results = bootstrap_answer_probability_tables_with_meta(
        old_results = test_results,
        N_samples_new = 100,
        seed = 124 + rp_idx,
    )
    
    prob_thre = 0.95
    print("Num. of repeat:",rp_idx)
    start_L2 = time.time()
    tables_by_llm_L2 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 2)
    tables_by_llm_L2_all.append(tables_by_llm_L2)
    end_L2 = time.time()
    print('L=2 finished! Time=',(end_L2-start_L2)/60,'mins')
    tables_by_llm_L3 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 3)
    tables_by_llm_L3_all.append(tables_by_llm_L3)
    end_L3 = time.time()
    print('L=3 finished! Time=',(end_L3-end_L2)/60,'mins')
    tables_by_llm_L4 = run_stop_per_llm_known(train_results=train_results,test_results=new_test_results, prob_thre=prob_thre, L_approx = 4)
    tables_by_llm_L4_all.append(tables_by_llm_L4)
    end_L4 = time.time()
    print('L=4 finished! Time=',(end_L4-end_L3)/60,'mins')
    tables_by_llm_ACbeta = run_AC_beta_per_llm(test_results, conf_thre=prob_thre)
    tables_by_llm_ACbeta_all.append(tables_by_llm_ACbeta)
    

Num. of repeat: 0
Finished 40/250 | Time spent: 0.02 mins
Finished 80/250 | Time spent: 0.04 mins
Finished 120/250 | Time spent: 0.04 mins
Finished 160/250 | Time spent: 0.04 mins
Finished 200/250 | Time spent: 0.04 mins
Finished 240/250 | Time spent: 0.04 mins
L=2 finished! Time= 0.039382565021514895 mins
Finished 40/250 | Time spent: 0.09 mins
Finished 80/250 | Time spent: 0.15 mins
Finished 120/250 | Time spent: 0.15 mins
Finished 160/250 | Time spent: 0.15 mins
Finished 200/250 | Time spent: 0.15 mins
Finished 240/250 | Time spent: 0.15 mins
L=3 finished! Time= 0.14683084885279338 mins
Finished 40/250 | Time spent: 0.46 mins
Finished 80/250 | Time spent: 0.75 mins
Finished 120/250 | Time spent: 0.75 mins
Finished 160/250 | Time spent: 0.75 mins
Finished 200/250 | Time spent: 0.75 mins
Finished 240/250 | Time spent: 0.75 mins
L=4 finished! Time= 0.7500715692838033 mins
Num. of repeat: 1
Finished 40/250 | Time spent: 0.02 mins
Finished 80/250 | Time spent: 0.02 mins
Finished 120/250 

Finished 40/250 | Time spent: 0.04 mins
Finished 80/250 | Time spent: 0.04 mins
Finished 120/250 | Time spent: 0.04 mins
Finished 160/250 | Time spent: 0.04 mins
Finished 200/250 | Time spent: 0.04 mins
Finished 240/250 | Time spent: 0.04 mins
L=3 finished! Time= 0.040388834476470944 mins
Finished 40/250 | Time spent: 0.21 mins
Finished 80/250 | Time spent: 0.22 mins
Finished 120/250 | Time spent: 0.22 mins
Finished 160/250 | Time spent: 0.22 mins
Finished 200/250 | Time spent: 0.22 mins
Finished 240/250 | Time spent: 0.22 mins
L=4 finished! Time= 0.2190861185391744 mins
Num. of repeat: 10
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.01 mins
Finished 120/250 | Time spent: 0.01 mins
Finished 160/250 | Time spent: 0.01 mins
Finished 200/250 | Time spent: 0.01 mins
Finished 240/250 | Time spent: 0.01 mins
L=2 finished! Time= 0.006887483596801758 mins
Finished 40/250 | Time spent: 0.00 mins
Finished 80/250 | Time spent: 0.01 mins
Finished 120/250 | Time spent: 0.

Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.02 mins
Finished 120/250 | Time spent: 0.02 mins
Finished 160/250 | Time spent: 0.02 mins
Finished 200/250 | Time spent: 0.02 mins
Finished 240/250 | Time spent: 0.02 mins
L=4 finished! Time= 0.017551215489705403 mins
Num. of repeat: 19
Finished 40/250 | Time spent: 0.01 mins
Finished 80/250 | Time spent: 0.02 mins
Finished 120/250 | Time spent: 0.02 mins
Finished 160/250 | Time spent: 0.02 mins
Finished 200/250 | Time spent: 0.02 mins
Finished 240/250 | Time spent: 0.02 mins
L=2 finished! Time= 0.022949783007303874 mins
Finished 40/250 | Time spent: 0.05 mins
Finished 80/250 | Time spent: 0.05 mins
Finished 120/250 | Time spent: 0.05 mins
Finished 160/250 | Time spent: 0.05 mins
Finished 200/250 | Time spent: 0.05 mins
Finished 240/250 | Time spent: 0.05 mins
L=3 finished! Time= 0.05346796909968058 mins
Finished 40/250 | Time spent: 0.27 mins
Finished 80/250 | Time spent: 0.29 mins
Finished 120/250 | Time spent: 0

In [21]:
print("Prob. Threshold = ", prob_thre)
print_avg_metrics_rp(tables_by_llm_ACbeta_all, test_results, method="AC-Beta")
print_avg_metrics_rp(tables_by_llm_L2_all, test_results, L_approx=2)
print_avg_metrics_rp(tables_by_llm_L3_all, test_results, L_approx=3)

Prob. Threshold =  0.95
Average metrics per LLM (AC-Beta; rp=20):
Qwen72B25             mean n_stop = 4.524 ± 0.090    mode accuracy = 1.000 ± 0.000    answer accuracy = 0.952 ± 0.006

Average metrics per LLM (L=2; rp=20):
Qwen72B25             mean n_stop = 2.482 ± 0.275    mode accuracy = 0.995 ± 0.002    answer accuracy = 0.949 ± 0.006

Average metrics per LLM (L=3; rp=20):
Qwen72B25             mean n_stop = 2.383 ± 0.263    mode accuracy = 0.995 ± 0.002    answer accuracy = 0.949 ± 0.006

